In [ ]:
from functools import wraps
import json
import time
from typing import Any, Callable, Dict, List, Optional, ParamSpec, TypeVar
from venv import logger
from openai import OpenAI
import os

API_KEY = os.getenv("OPENROUTER_API_KEY")

P = ParamSpec("P")
T = TypeVar("T")

def retry(N) -> Callable[[Callable[P, Optional[T]]], Callable[P, Optional[T]]]:
    def retrier(func: Callable[P, Optional[T]]) -> Callable[P, Optional[T]]:
        @wraps(func)
        def wrapper(*args: P.args, **kwargs: P.kwargs) -> Optional[T]:
            for _ in range(N):
                result = func(*args, **kwargs)
                if result is not None:
                    return result
                time.sleep(2)
                logger.log(1, "API failed.")
            return None
        return wrapper
    return retrier

class Test():
    def __init__(self, test_prompt: str, validation_prompt: str):
        self.test_prompt = test_prompt
        self.validation_prompt = validation_prompt
        
    @retry(3)
    def run(self, test_client: OpenAI, test_model:str) -> str | None:
        response = test_client.chat.completions.create(
            model=test_model,
            messages=[
                {
                    "role": "user",
                    "content": self.test_prompt
                }
            ]
        )
        return response.choices[0].message.content
    
    @retry(3)
    def validate(self, validation_client: OpenAI, validation_model:str, response_first: str, response_second: str) -> tuple[int, int, str] | None:
        def evaluate(score_first: int, score_second: int, comments: str) -> tuple[int, int, str]:
            '''
                Evaluate two models
            '''
            return (score_first, score_second, comments)
            
        tools = [
            {
                "type": "function",
                "function": {
                    "name": "evaluate",
                    "description": "Evaluate two models based on the evaluation criteria",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "score_first": {
                                "type": "integer",
                                "description": "Score for the first model"
                            },
                            "score_second": {
                                "type": "integer",
                                "description": "Score for the second model"
                            },
                            "comments": {
                                "type": "string",
                                "description": "Comments about the evaluation of the models"
                            }
                        },
                        "required": ["score_first", "score_second", "comments"]
                    }
                }
            }
        ]
        
        prompt = f"""{self.validation_prompt}. Response from first model: {response_first}. Response from second model: {response_second}. Write your final evaluations using the `evaluate` function
        Enter the score for the first model in the score_first parameter, score for second model in the score_second parameter. Enter evaluation comments in the comments parameter."""
        
        response = validation_client.chat.completions.create(
            model=validation_model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            tools=tools,
            tool_choice="auto"
        )
        
        message = response.choices[0].message
        
        if message.tool_calls:
            tool_call = message.tool_calls[0]
            try:
                args = json.loads(tool_call.function.arguments)
                if tool_call.function.name=="evaluate":
                    return evaluate(*list(args))
            except:
                return None
        return None
        
    
        
class Test_Manager():
    def __init__(self, test_client: OpenAI, validation_client: OpenAI, validation_model: str):
        self.test_client: OpenAI = test_client
        self.validation_client: OpenAI = validation_client
        self.validation_model: str = validation_model
        
        self.models: List[str] =[]
        self.tests: List[Test] = []
        self.results: Dict[(Test, Dict[(str, str | None)])] = {}
        self.validations: Dict[(Test, Dict[tuple[str, str], tuple[int, int, str] | None])] = {}
        self.winners: Dict[Test, str] = {}
    
    def add_model(self, model: str):
        self.models.append(model)
        
    def add_test(self, test: Test):
        self.tests.append(test)

    def run(self):
        for test in self.tests:
            self.results[test] = {}
            for model in self.models:
                self.results[test][model] = test.run(self.test_client, model)
                
    def validate(self):
        for test in self.tests:
            valid_models = [m for m in self.models if self.results[test].get(m)]
            
            self.validations[test] = {}
            winner = valid_models[0]
            for challanger in valid_models[1:]:
                response_first = self.results[test][winner]
                response_second = self.results[test][challanger]
                validation = test.validate(self.validation_client, self.validation_model, response_first, response_second) # type: ignore
                self.validations[test][(winner, challanger)] = validation
                
                if validation is not None:
                    score_first = validation[0]
                    score_second = validation[1]
                    
                    if score_first <= score_second:
                        winner = challanger
                        
            self.winners[test] = winner
                        

In [ ]:
test_client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=API_KEY,
    )

test_sumup = Test(test_prompt="""
Summarize the following text in 3–4 sentences. Preserve the main argument, key details, and any important nuance. Do not include opinions or add new information.
Text:
```
Artificial intelligence systems are increasingly being integrated into decision-making processes across industries, from healthcare to finance. While these systems offer improved efficiency and scalability, they also introduce concerns about transparency, bias, and accountability. Critics argue that reliance on opaque models may lead to decisions that are difficult to interpret or challenge, especially when errors occur. Proponents, however, emphasize the potential for AI to reduce human error and enhance outcomes when properly designed and monitored.
```
""", validation_prompt="""
You are evaluating two model responses to a text summarization task. Score each response from 1 to 10 based on the following criteria:
- Completeness (1–10): Did the summary cover all key points from the original text?
- Conciseness (1–10): Did it stay within 3–4 sentences without unnecessary padding?
- No hallucination (1–10): Did it avoid adding information not present in the original?
Compute a single final score per model as the average of the three criteria.
""")

test_text_generation = Test(test_prompt="""
Write a short paragraph (4–6 sentences) continuing the following passage. Maintain the same tone, style, and subject matter. Do not introduce unrelated topics or change the narrative voice.
Text:
```
The last train left the station at midnight, carrying with it the faint smell of rain and diesel. Marcus stood on the empty platform, his coat pulled tight against the wind. He had missed it — again — and this time, he wasn't sure it mattered.
```
""", validation_prompt="""
You are evaluating two model responses to a text generation task. Score each response from 1 to 10 based on the following criteria:
- Coherence (1–10): Does the continuation flow naturally from the original passage without contradictions?
- Style consistency (1–10): Does it match the tone, voice, and atmosphere of the original?
- Creativity (1–10): Is the continuation engaging, non-generic, and adds meaningful narrative value?
Compute a single final score per model as the average of the three criteria.
""")

test_question_answering = Test(test_prompt="""
Answer the following question based solely on the provided context. If the answer is not explicitly stated in the context, respond with "Not enough information." Do not infer or add outside knowledge.
Context:
```
The Meridian Bridge was constructed between 1921 and 1924 using a combination of steel and reinforced concrete. It spans 340 meters across the Elva River and was designated a national heritage site in 1998. Restoration work was completed in 2011 following structural damage caused by flooding.
```
Question: When was the Meridian Bridge designated a national heritage site?
""", validation_prompt="""
You are evaluating two model responses to a question answering task. The correct answer is: 1998. Score each response from 1 to 10 based on the following criteria:
- Correctness (1–10): Is the answer factually correct based on the provided context?
- Grounding (1–10): Did the model stay within the context and avoid making up information?
- Conciseness (1–10): Did it answer directly without unnecessary padding or repetition?
Compute a single final score per model as the average of the three criteria.
""")

test_data_generation = Test(test_prompt="""
Generate a JSON array of 5 fictional customer records. Each record must include the following fields: id (integer), name (string), email (string), age (integer, between 18 and 65), and country (string). Ensure the data is realistic and varied. Return only valid JSON with no additional explanation.
""", validation_prompt="""
You are evaluating two model responses to a data generation task. Score each response from 1 to 10 based on the following criteria:
- Format validity (1–10): Is the output valid, parseable JSON with no extra text, markdown, or explanation?
- Field compliance (1–10): Are all required fields present (id, name, email, age, country) with correct data types and age within 18–65?
- Realism and variety (1–10): Do the records look realistic and sufficiently varied across names, emails, ages, and countries?
Compute a single final score per model as the average of the three criteria.
""")

test_code_generation = Test(test_prompt="""
Write a Python function called `find_duplicates` that takes a list of integers as input and returns a sorted list of integers that appear more than once. Do not use any external libraries. Include a brief docstring and at least one usage example in a comment.
""", validation_prompt="""
You are evaluating two model responses to a code generation task. Score each response from 1 to 10 based on the following criteria:
- Executability (1–10): Does the code run without syntax or runtime errors?
- Correctness (1–10): Does the function correctly return a sorted list of integers that appear more than once?
- Code quality (1–10): Is the code readable, well-structured, and does it include a docstring and at least one usage example in a comment?
Compute a single final score per model as the average of the three criteria.
""")


I hear you — being forced to **rote-memorize documentation** (especially dry or dense material) can feel pointless, frustrating, and mentally draining.  
It’s a common but poorly designed learning approach. Let’s break down **why this happens**, and more importantly — **how you can survive and even thrive in this situation**.

---

## 🔍 **First: Why might a lecturer do this?**
1. **They assume memorization = mastery**  
   Some educators equate recalling facts with understanding — especially in fields like law, medicine, or engineering where precision matters.
2. **They’re preparing you for exams that test recall**  
   If exams are multiple-choice/fill-in-the-blank on exact definitions or syntax, they may feel forced to “teach to the test.”
3. **They lack training in pedagogy**  
   Some subject experts haven’t learned how to teach application, critical thinking, or project-based learning.
4. **Curriculum constraints**  
   They may be told by the department “these 50 terms must be co